In [1]:
from langgraph.graph import StateGraph,START,END

In [ ]:
from typing import TypedDict
from dotenv import load_dotenv
from langchain_xai import ChatXAI

C:\Users\Archi\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
%pip install -qU langchain-xai python-dotenv

In [ ]:
load_dotenv()

In [ ]:


model = ChatXAI(
    model="grok-4",  # or "grok-3-mini", "grok-3-fast", etc.
    api_key='GROQ_API_KEY',
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
)

In [ ]:
class parallelstate(TypedDict):
    balls:int
    fours:int
    six:int

    runs:int
    strike_rate:float
    ball_per_boundary:float
    runs_from_boundary:int


In [ ]:
graph = StateGraph(parallelstate)

In [ ]:
def strike_rate(state:parallelstate):
    balls=state['balls']
    strike_r = state['runs']/balls * 100
    state['strike_rate'] = strike_r
    return {'strike_rate':strike_r}



In [ ]:
def bpb(state:parallelstate):
     balls=state['balls']
     bpg = balls/(state['fours']+state['six'])
     state['ball_per_boundary'] = bpg
     return {'ball_per_boundary':bpg}


In [ ]:
def rfb(state:parallelstate):
    runs_from_boundary = (state['fours']*4 ) + ( state['six']*6)
    state['runs_from_boundary'] = runs_from_boundary
    return {'runs_from_boundary':runs_from_boundary }

In [ ]:
def summary(state:parallelstate)->parallelstate:
    print(state['ball_per_boundary'],
          state['strike_rate'],
          state['runs_from_boundary'])
    return state


In [ ]:
graph.add_node("calculate_strike_rate",strike_rate)



In [ ]:
graph.add_node('calculate_boundary_per_ball',bpb)
graph.add_node('calculate_runs_from_boundary',rfb)
graph.add_node('summary',summary)

In [ ]:
graph.add_edge('calculate_strike_rate','summary')
graph.add_edge('calculate_boundary_per_ball','summary')
graph.add_edge('calculate_runs_from_boundary','summary')
graph.add_edge(START,'calculate_strike_rate')
graph.add_edge(START,'calculate_boundary_per_ball')
graph.add_edge(START,'calculate_runs_from_boundary')
graph.add_edge('summary',END)


In [ ]:
workflow=graph.compile()

In [ ]:
initial_state= {
    'runs':35,
    'six':2,
    'fours':3,
    'balls':23


}

In [ ]:
result=workflow.invoke(initial_state)
print(result)